# Hand-designed image features: gradients, HOG, and SIFT

_Feature Engineering_

**Before deep nets, vision turned raw pixels into edge-orientation histograms (HOG, SIFT) to get robust shape features.**

An image is a grid of pixel brightness values. The obvious "feature" is just the raw
       pixels &mdash; flatten the grid into one long vector and hand it to a model. Chapter 8 of
       Zheng & Casari's Feature Engineering for Machine Learning opens by explaining why that
       is a terrible feature. The very same object photographed a little brighter, shifted a few
       pixels right, or scaled up looks like a completely different vector of numbers, even though it is
       the same thing. Raw pixels are fragile: sensitive to lighting, shift, and scale.

       So for decades &mdash; the whole era before deep learning &mdash; people hand-designed
       smarter image features. The trick they converged on: don't describe an image by its pixel
       brightness, describe it by its gradients. A gradient is how fast brightness changes
       and in which direction. Gradients are large exactly at edges (where one region meets
       another) and they barely move when you make the whole image brighter or darker. Edges are where
       the shape lives.

---

This notebook is a practice scaffold for the **Hand-designed image features: gradients, HOG, and SIFT** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# fe-image-features.py
# LESSON: Hand-crafted image features from gradients (HOG — Histogram of Oriented
# Gradients) vs RAW PIXELS.
#
# PROBLEM we reproduce: a classifier on RAW PIXEL intensities is BRITTLE to small
# spatial SHIFTS. A pixel value means "how bright is location (r, c)". If you slide
# the whole digit over by 1-2 pixels, every pixel value lands in a different column,
# so the feature vector changes a lot even though it is the SAME digit. We train on
# the original load_digits images, then test on SHIFTED copies -> raw-pixel accuracy
# DROPS a lot.
# FIX: describe each image by a gradient-ORIENTATION histogram (a simple HOG). We do
# NOT store brightness per location; we store "how much edge energy points in each
# DIRECTION", pooled over a few cells. Edge directions barely change when you nudge
# the image, so the SAME classifier holds up much better on the shifted test set.
#
# Runs top-to-bottom in Colab. Packages: numpy, matplotlib, scikit-learn only
# (we hand-roll HOG in numpy — no skimage).

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------------------
# STEP 1: Load REAL data — sklearn's bundled handwritten digits (8x8 grayscale).
# Each image is an 8x8 array of pixel intensities (0..16). No download needed.
# ---------------------------------------------------------------------------
digits = load_digits()
images = digits.images                       # shape (1797, 8, 8)
y = digits.target                            # the digit 0..9 for each image
X_pixels = images.reshape(len(images), -1)   # flatten to 64 raw-pixel features

# Fixed split so the numbers reproduce.
idx = np.arange(len(images))
idx_tr, idx_te = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)


def shift_image(img, dr, dc):
    """Slide an 8x8 image by (dr, dc) pixels using np.roll (wrap-around)."""
    return np.roll(np.roll(img, dr, axis=0), dc, axis=1)


def hog_features(img, n_bins=9):
    """Simple HOG in pure numpy.

    1. np.gradient -> how brightness changes down (gy) and across (gx).
    2. magnitude  = sqrt(gx^2 + gy^2)  -> how strong the edge is at each pixel.
    3. orientation = arctan2(gy, gx)   -> which DIRECTION the edge points.
    4. Split the 8x8 image into 2x2 = 4 cells. In each cell, build a histogram
       over orientation, where each pixel votes with its magnitude (strong edges
       count more). Concatenate the 4 cell histograms -> the feature vector.
    """
    gy, gx = np.gradient(img.astype(float))          # vertical & horizontal change
    mag = np.sqrt(gx ** 2 + gy ** 2)                 # edge strength per pixel
    ang = np.arctan2(gy, gx)                         # edge direction, in [-pi, pi]
    ang = (ang + np.pi) % np.pi                      # fold to [0, pi): edges are undirected
    feats = []
    half = img.shape[0] // 2                         # 8 -> cells of size 4x4
    bin_edges = np.linspace(0, np.pi, n_bins + 1)
    for r0 in (0, half):                             # loop over the 2x2 grid of cells
        for c0 in (0, half):
            cell_ang = ang[r0:r0 + half, c0:c0 + half].ravel()
            cell_mag = mag[r0:r0 + half, c0:c0 + half].ravel()
            # magnitude-weighted histogram of orientations in this cell
            hist, _ = np.histogram(cell_ang, bins=bin_edges, weights=cell_mag)
            feats.append(hist)
    h = np.concatenate(feats)
    return h / (np.linalg.norm(h) + 1e-6)            # normalize -> contrast-robust


# ---------------------------------------------------------------------------
# STEP 2: Visualize the PURE (raw) data — one digit image, plus the HOG view of it.
# ---------------------------------------------------------------------------
sample = images[idx_te[0]]
gy, gx = np.gradient(sample.astype(float))
sample_ang = (np.arctan2(gy, gx) + np.pi) % np.pi
sample_mag = np.sqrt(gx ** 2 + gy ** 2)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(sample, cmap="gray_r")
ax[0].set_title(f"STEP 2: raw digit image (label={y[idx_te[0]]})")
ax[0].set_xlabel("column (pixel position)"); ax[0].set_ylabel("row")
# gradient-orientation histogram of the WHOLE image (magnitude-weighted)
whole_hist, edges = np.histogram(sample_ang.ravel(), bins=np.linspace(0, np.pi, 10),
                                 weights=sample_mag.ravel())
centers = (edges[:-1] + edges[1:]) / 2
ax[1].bar(centers, whole_hist, width=(np.pi / 9) * 0.9, color="steelblue")
ax[1].set_title("STEP 4 preview: gradient-orientation histogram (HOG)")
ax[1].set_xlabel("edge direction (radians, 0..pi)"); ax[1].set_ylabel("magnitude-weighted votes")
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# STEP 3: Reproduce the PROBLEM — train on ORIGINAL pixels, test on SHIFTED pixels.
# We shift each test image by (1, 1) pixels. Raw-pixel features change a lot.
# ---------------------------------------------------------------------------
shifted_images = np.array([shift_image(im, 1, 1) for im in images])
X_pixels_shift = shifted_images.reshape(len(images), -1)

pix_clf = LogisticRegression(max_iter=5000)
pix_clf.fit(X_pixels[idx_tr], y[idx_tr])                 # train on clean pixels
pix_acc_clean = accuracy_score(y[idx_te], pix_clf.predict(X_pixels[idx_te]))
pix_acc_shift = accuracy_score(y[idx_te], pix_clf.predict(X_pixels_shift[idx_te]))

# ---------------------------------------------------------------------------
# STEP 4: Apply the FIX — turn every image into HOG features, then visualize.
# (The per-image HOG bar chart above already showed what one looks like.)
# ---------------------------------------------------------------------------
X_hog = np.array([hog_features(im) for im in images])
X_hog_shift = np.array([hog_features(im) for im in shifted_images])

# ---------------------------------------------------------------------------
# STEP 5: Show the FIX — the SAME classifier on HOG features, tested on SHIFTED data.
# ---------------------------------------------------------------------------
hog_clf = LogisticRegression(max_iter=5000)
hog_clf.fit(X_hog[idx_tr], y[idx_tr])                    # train on clean HOG
hog_acc_clean = accuracy_score(y[idx_te], hog_clf.predict(X_hog[idx_te]))
hog_acc_shift = accuracy_score(y[idx_te], hog_clf.predict(X_hog_shift[idx_te]))

# Bar chart: accuracy on SHIFTED test data, raw pixels vs HOG.
fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.bar(["raw pixels", "HOG features"], [pix_acc_shift, hog_acc_shift],
        color=["indianred", "seagreen"])
ax2.set_title("STEP 5: accuracy on SHIFTED test images")
ax2.set_ylabel("accuracy"); ax2.set_ylim(0, 1.05)
for i, v in enumerate([pix_acc_shift, hog_acc_shift]):
    ax2.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()

print(f"raw pixels:   clean test acc = {pix_acc_clean:.3f}   shifted test acc = {pix_acc_shift:.3f}")
print(f"HOG features: clean test acc = {hog_acc_clean:.3f}   shifted test acc = {hog_acc_shift:.3f}")
print(f"PROBLEM (raw): {pix_acc_shift:.3f}   →   FIX (engineered): {hog_acc_shift:.3f}")

## Reference implementation — scikit-image + OpenCV (numpy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import color, data

# --- A sample image (use any image; the book uses its own image datasets:
#     github.com/alicezheng/feature-engineering-book) ---
image = color.rgb2gray(data.astronaut())   # grayscale: HOG works on intensity

# === HOG: Histogram of Oriented Gradients ===
# - compute the gradient (magnitude & orientation) at each pixel
# - split the image into 16x16-pixel cells
# - build an 8-bin orientation histogram per cell
# - block-normalize over 1x1 blocks of cells
# visualize=True also returns an image showing the dominant edge directions.
features, hog_image = hog(
    image,
    orientations=8,
    pixels_per_cell=(16, 16),
    cells_per_block=(1, 1),
    visualize=True
)

print('HOG feature vector length:', features.shape)   # one long vector of cell histograms

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(image, cmap='gray');      ax1.set_title('input image')
ax2.imshow(hog_image, cmap='gray');  ax2.set_title('HOG: dominant edge directions')
plt.show()

# === SIFT: Scale-Invariant Feature Transform (OpenCV) ===
import cv2
gray8 = (image * 255).astype(np.uint8)     # OpenCV wants an 8-bit image

sift = cv2.SIFT_create()                   # the SIFT detector/descriptor
keypoints, descriptors = sift.detectAndCompute(gray8, None)
# keypoints: detected across SCALES (scale & rotation invariant)
# descriptors: one 128-d orientation-histogram descriptor per keypoint
print('SIFT keypoints:', len(keypoints), ' descriptor shape:', descriptors.shape)

## Visualize the data & results

_Take a real 8x8 digit image, compute its pixel gradients, and bin the gradient ORIENTATIONS into an 8-bin histogram (the core of HOG). Which edge directions dominate this digit?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

d = load_digits()
img = d.images[0]                 # one real 8x8 image (digit '0'), pixel values 0..16

# --- gradients at every pixel (numpy only) ---
gy, gx = np.gradient(img.astype(float))      # change down (gy) and right (gx)
mag = np.sqrt(gx**2 + gy**2)                 # edge strength per pixel
ang = np.arctan2(gy, gx)                     # edge direction, in (-pi, pi]
ang = np.mod(np.degrees(ang), 360.0)         # fold to [0, 360) degrees

# --- bin orientations into 8 wedges, weighted by magnitude (HOG's core) ---
nbins = 8
bin_idx = np.minimum((ang / (360.0 / nbins)).astype(int), nbins - 1)
hist = np.zeros(nbins)
for k in range(nbins):
    hist[k] = mag[bin_idx == k].sum()

print('orientation histogram:', np.round(hist, 1))
# -> [78.2 27.1 35.7 25.9 77.8 41.9 24.5 26. ]
# near-horizontal bins dominate: the round '0' has strong left/right-pointing gradients.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why is a flattened vector of raw pixel brightnesses a poor image feature, and what single idea fixes most of the problem?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Brighten the whole image, or shift the object two pixels right. — _Every raw pixel value changes, so the feature vector changes drastically even though it is the same object._
- Switch from brightness to gradients (differences of neighboring pixels). — _A constant brightness offset cancels in a difference, so gradients ignore additive lighting; gradients are large only at edges, which carry the shape._
- Pool gradient orientations into per-cell histograms (HOG). — _A small shift keeps edges in the same cell and bin, so the histogram is stable &mdash; local shift and lighting tolerance._

**Answer:** Raw pixels are fragile: lighting, shift, and scale change every value, so the same object yields very different vectors. The fix is to describe the image by its gradients (edge strength and direction) instead of brightness, then pool gradient orientations into histograms over cells &mdash; the core of HOG. Gradients ignore additive lighting and pooling absorbs small shifts.

</details>

**Problem 2.** An 8&times;8 HOG cell uses 8 orientation bins. Most of its gradient magnitude lands in the bin covering 80&deg;&ndash;100&deg;. What does that tell you about the cell's content?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the gradient points across an edge, perpendicular to the edge line. — _A gradient near 90 degrees (pointing up/down) means brightness changes vertically._
- Brightness changing vertically corresponds to a horizontal edge. — _The edge line itself runs left-to-right; the gradient crosses it pointing up or down._
- A large histogram value in that bin means strong, consistent edges of that orientation. — _The histogram sums magnitudes, so a tall bin means lots of edge energy in that direction._

**Answer:** The cell is dominated by a strong horizontal edge. The gradient orientation near $90\degree$ points vertically (brightness changes top-to-bottom), which is exactly what a horizontal boundary produces. A tall bin means lots of edge magnitude in that direction &mdash; HOG has summarized the cell as "mostly horizontal edge here".

</details>

**Problem 3.** You photograph the same logo small in one image and large (and slightly rotated) in another. Plain HOG on a fixed grid struggles to match them. What does SIFT add to handle this, and why does it work?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note HOG runs on a fixed cell grid at one scale, so a size change moves everything to different cells. — _The descriptor isn't aligned across the two images, so the vectors don't match._
- SIFT detects keypoints across a scale-space (the image repeatedly blurred and shrunk). — _The same physical point is found whether the logo is near or far &mdash; scale invariance._
- SIFT measures each keypoint's dominant gradient direction and rotates its orientation histograms to that reference. — _A rotation of the image just rotates the reference too, so the descriptor is unchanged &mdash; rotation invariance._

**Answer:** SIFT adds scale and rotation invariance. It searches a scale-space (image blurred/shrunk repeatedly) for keypoints that persist across scales, so the same point is found at any size; and it rotates each keypoint's orientation histograms to a dominant direction first, so rotation doesn't change the descriptor. Both still rest on the same gradient-orientation-histogram idea as HOG &mdash; SIFT just makes it scale- and rotation-invariant.

</details>